# Esep safe-6 — public portfolio reproduction

Enable GPU T4 x2 and Internet; use a fresh Kaggle session and Save & Run All. No GitHub token required. This packaging has not had a new GPU validation. Historical results are in the repository evidence folder.


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/esep-soup-kernels')
if repo.exists():
    raise RuntimeError('Use a fresh Kaggle session to preserve prior outputs.')
subprocess.run(['git', 'clone', '--branch', 'v1.0.0', '--depth', '1', 'https://github.com/gojosatorukz/esep-soup-kernels.git', str(repo)], check=True)
def run(*args, **kwargs):
    subprocess.run(list(args), cwd=repo, check=True, **kwargs)
run(sys.executable, 'scripts/preflight.py', '--system-only')
run(sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu128')
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt')
run(sys.executable, 'scripts/preflight.py')
run(sys.executable, 'scripts/verify_source.py')


In [ ]:
import os
run(sys.executable, 'task/prepare_data.py')
run(sys.executable, '-m', 'pytest', 'solution', '-q')
run(sys.executable, '-m', 'pytest', 'tests', '-q', env={**os.environ, 'SOUP_APPLY_SOLUTION':'1'})
run(sys.executable, 'scripts/verify_source.py')


In [ ]:
run(sys.executable, 'harness/run_task.py', '--repeats', '2')
run(sys.executable, 'harness/check_rules.py', '--result', 'runs/result.json')
run(sys.executable, 'harness/score.py', 'runs/result.json')
run(sys.executable, 'scripts/verify_source.py')


In [ ]:
import json
result = json.loads((repo/'runs/result.json').read_text())
checks = json.loads((repo/'runs/checks.json').read_text())
import hashlib
if checks.get('result_sha256') != hashlib.sha256((repo/'runs/result.json').read_bytes()).hexdigest():
    raise RuntimeError('Checks do not match the generated result file.')
if len(checks['checks']) != 10 or not all(row['ok'] for row in checks['checks']):
    raise RuntimeError('At least one harness check failed.')
print(json.dumps(result.get('summary', {}), indent=2))
print('All ten harness checks passed. Download the runs/ directory from this session.')
